# Deploying NVIDIA Nemotron 3.5 Lightning with vLLM

This notebook will walk you through how to run the NVIDIA Nemotron 3.5 Lightning NVFP4 checkpoint with vLLM on a single H100.

[vLLM](https://docs.vllm.ai) is a fast and easy-to-use library for LLM inference and serving.

Nemotron 3.5 Lightning is published as two checkpoints:

- **BF16** — [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16)
- **NVFP4** — [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4) - **this notebook deploys this checkpoint**

**Model size:** 30B total parameters, 3B active (MoE)

Prerequisites for this notebook:
- 1x NVIDIA H100 80GB with recent drivers
- [Docker](https://docs.docker.com/engine/install/) with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html)
- Python 3.10+

## Overview

- **Serve** the Nemotron 3.5 Lightning NVFP4 checkpoint using vLLM
- **Query the model** through an OpenAI-compatible API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget

## Table of Contents

1. **Environment setup** - Dependencies and container image
2. **Verify GPU** - Confirm CUDA and GPU availability
3. **OpenAI-compatible server** - Launch and query vLLM
   - **Start server** - Base, MTP, DFlash, or DSpark configuration
   - **Generate responses** - Single, sequential, and streamed completions
   - **Reasoning** - Toggle thinking on/off
   - **Tool calling** - Function calling via OpenAI tools schema
   - **Controlling Reasoning Budget** - Limit reasoning trace length
4. **Cleanup and shutdown**

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (1x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Haw6pwtZwgoiPRoO2R4cbiAASY)

## Environment setup

### Pull the vLLM Docker image

The model runs inside a vLLM container. Pull it once before starting the server:

```shell
docker pull vllm/vllm-openai:v0.27.1
```

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `torch` backs the GPU check. The container already carries everything the model needs in order to load and run.

> **Note:** Match the `torch` build to your driver. A CUDA 13 wheel on a CUDA 12 driver reports `CUDA available: False` even when the GPU is healthy. In case that happens, check the driver's CUDA version with `nvidia-smi` and install from the matching index if needed, for example `--index-url https://download.pytorch.org/whl/cu128`. This affects only the GPU check below, not the model serving.

In [1]:
# Ensure pip is available
!python3 -m ensurepip --default-pip

Looking in links: /tmp/tmpuynnmwxt


In [ ]:
%pip install openai==2.38.0 transformers==5.9.0 torch

## Verify GPU

Confirm your GPU is visible on the host before starting the Docker container.

> **Expected output:** `CUDA available: True` with one H100 listed. If CUDA is `False`, check your driver installation.

In [1]:
# GPU environment check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 1
GPU[0]: NVIDIA H100 80GB HBM3


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using vLLM.

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the vLLM container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it --gpus all --ipc=host --network=host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  --entrypoint /bin/bash \
  vllm/vllm-openai:v0.27.1
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

All `vllm serve` commands below should be run from inside this container.

### Configuration reference

| | NVFP4 |
|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` |
| **Hardware (this notebook)** | 1x H100 80GB |
| **Docker image** | `vllm/vllm-openai:v0.27.1` |
| **Quantization** | `modelopt_mixed` (auto-detected from the checkpoint) |
| **MoE backend** | `humming` |
| **Linear backend** | `humming` (base configuration) |
| **Mamba backend** | `flashinfer` with FP16 SSM cache |
| **Max model length** | 65536 |
| **Max batched tokens** | 32768 |
| **Max concurrent sequences** | 256 base, 128 with speculative decoding |
| **Speculative decoding** | None, MTP, DFlash, or DSpark (3 draft tokens) |
| **Reasoning parser** | `nemotron_v3` |
| **Tool parser** | `qwen3_coder` |
| **Port** | 8000 |

### Start server

Run one of the following commands from inside the Docker container terminal. All four serve the same NVFP4 checkpoint and differ only in speculative decoding. Every later cell in this notebook works with any of them.

> **Note:** Parser names are backend-specific and not interchangeable. vLLM uses `--reasoning-parser nemotron_v3` and `--tool-call-parser qwen3_coder`. TensorRT-LLM uses `--reasoning_parser nemotron-v3` and SGLang uses `--reasoning-parser nemotron_3` — these are different identifiers for the same logical capability.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --moe-backend humming \
  --linear-backend humming \
  --max-num-seqs 256 \
  --max-model-len 65536 \
  --max-num-batched-tokens 32768 \
  --enable-prefix-caching \
  --async-scheduling \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --mamba-cache-mode align \
  --mamba-ssu-algorithm horizontal \
  --reasoning-parser nemotron_v3 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --host 127.0.0.1 \
  --port 8000
```

#### MTP

MTP (Multi-Token Prediction) uses a small draft layer built into the model to guess several tokens ahead, which the model then verifies in one step.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --moe-backend humming \
  --max-num-seqs 128 \
  --max-model-len 65536 \
  --max-num-batched-tokens 32768 \
  --enable-prefix-caching \
  --async-scheduling \
  --speculative_config.method mtp \
  --speculative_config.num_speculative_tokens 3 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --mamba-cache-mode align \
  --reasoning-parser nemotron_v3 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --host 127.0.0.1 \
  --port 8000
```

#### DFlash

DFlash does the same thing, but the guesses come from a separate draft model rather than a layer inside the model.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --moe-backend humming \
  --max-num-seqs 128 \
  --max-model-len 65536 \
  --max-num-batched-tokens 32768 \
  --enable-prefix-caching \
  --async-scheduling \
  --speculative_config.method dflash \
  --speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
  --speculative_config.num_speculative_tokens 3 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --mamba-cache-mode align \
  --reasoning-parser nemotron_v3 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --host 127.0.0.1 \
  --port 8000
```

#### DSpark

DSpark also uses a separate draft model, but it guesses a whole block of tokens at once rather than one at a time.

> **Note:** `--speculative_config.num_speculative_tokens` must be at least the draft checkpoint's `dspark_block_size`. vLLM refuses to start below that, because a shorter block produces garbled output rather than just fewer accepted tokens.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --moe-backend humming \
  --max-num-seqs 128 \
  --max-model-len 65536 \
  --max-num-batched-tokens 32768 \
  --enable-prefix-caching \
  --async-scheduling \
  --speculative_config.method dspark \
  --speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
  --speculative_config.num_speculative_tokens 3 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --mamba-cache-mode align \
  --reasoning-parser nemotron_v3 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --host 127.0.0.1 \
  --port 8000
```

### Wait for the server to be ready

Before sending any requests, confirm the server is up by polling `/v1/models`.

Run the following in the same terminal (or a separate one):

```shell
until curl -sf http://localhost:8000/v1/models > /dev/null 2>&1; do
  echo "Waiting for server..."; sleep 5
done
echo "Server is ready"
```

> **Expected output:** Once the server is ready, the loop exits and prints `Server is ready`. You can also run `curl http://localhost:8000/v1/models` directly to see the list of loaded model IDs.

> **Note:** On the first run, the model weights will be downloaded from Hugging Face before loading begins, so the combined download and load time will be longer than on subsequent runs.

### Generate responses

The cells below show single, sequential, and streamed completions, followed by reasoning on/off, tool calling, and reasoning budget examples.

> **Note:** Reasoning tokens count toward `max_tokens`. If `content` comes back empty or `None`, the reasoning trace consumed the entire budget before the model produced an answer — raise `max_tokens`.

In [1]:
from openai import OpenAI

# Set this to match the --served-model-name used when starting the server
SERVED_MODEL_NAME = "nemotron-3.5-lightning"
BASE_URL = "http://127.0.0.1:8000/v1"

client = OpenAI(base_url=BASE_URL, api_key="null")

In [2]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Briefly explain: what is vLLM and why is it useful for large model inference?"},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
)
choice = response.choices[0]
print("Reasoning:", choice.message.reasoning)
print("Content:", choice.message.content)

Reasoning: Here's a thinking process:

1.  **Analyze User Input:**
   - User wants a brief explanation of what vLLM is and why it's useful for large model inference.
   - Key elements: definition of vLLM, its usefulness/benefits for large model inference.

2.  **Identify Core Concepts of vLLM:**
   - vLLM is an open-source library for efficient serving of large language models.
   - Key innovation: PagedAttention (memory-efficient attention mechanism).
   - Solves the key problem of KV cache memory usage.
   - Enables higher throughput, lower latency, better GPU utilization.
   - Developed by UC Berkeley SkyLab.

3.  **Structure the Answer:**
   - Definition: What is vLLM? (Open-source, fast, efficient LLM serving)
   - Key Innovation: PagedAttention
   - Why it's useful: Throughput, latency, GPU utilization, cost-effectiveness, ease of use.
   - Brief comparison/context (optional but helpful).
   - Keep it concise as requested ("Briefly explain").

4.  **Draft - Mental Refinement:**
 

### Sequential completions

Send multiple prompts in sequence and collect all responses.

In [4]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=2048,
    )
    print(f"Q: {prompt}")
    print(f"A: {response.choices[0].message.content}\n")

Q: What is the square root of 144?
A: The square root of 144 is **12** (since 12 × 12 = 144). 

Note: In mathematics, every positive number has two square roots (a positive and a negative), so -12 is also a square root of 144, but the principal (positive) square root is 12.

Q: What is the capital of France?
A: The capital of France is **Paris**.

Q: Explain quantum computing in simple terms.
A: ### Quantum Computing in Simple Terms

**Classical computers** use bits: tiny switches that are either **off (0)** or **on (1)**. They process information by flipping these switches one step at a time, or many at once using multiple cores.

**Quantum computers** use **qubits** (quantum bits). Thanks to the physics of tiny particles (like electrons or photons), qubits can do something bits can't: they can exist in a **superposition** of both 0 and 1 at the same time.

#### Two key ideas, simply:

1. **Superposition – The Spinning Coin**  
   Imagine a coin lying flat: it's either heads (0) or ta

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [5]:
# Streaming chat completion
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    stream=True,
)

section = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

Reasoning: Here's a thinking process:

1.  **Analyze User Input**: User asks "What are the first 5 prime numbers?"
2.  **Identify Core Concept**: Prime numbers are natural numbers greater than 1 that have no positive divisors other than 1 and themselves.
3.  **Recall/Determine First 5 Primes**: 
   - 2 (smallest prime, only even prime)
   - 3
   - 5
   - 7
   - 11
   Let me verify: 
   - 2: prime ✓
   - 3: prime ✓
   - 4: not prime (divisible by 2)
   - 5: prime ✓
   - 6: not prime (divisible by 2,3)
   - 7: prime ✓
   - 8,9,10: not prime
   - 11: prime ✓
   So first 5 are 2, 3, 5, 7, 11.
4.  **Formulate Response**: Simply list them, maybe with a brief definition or confirmation. Keep it concise.
   Output: The first 5 prime numbers are 2, 3, 5, 7, and 11.

Content: The first 5 prime numbers are:

**2, 3, 5, 7, 11**

(Prime numbers are natural numbers greater than 1 that have no positive divisors other than 1 and themselves.)

### Reasoning

The model supports two modes — **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`. Use `temperature=1.0, top_p=0.95` when reasoning is on, and `temperature=0.2` when it is off.

In [6]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
)
print("Reasoning:", resp.choices[0].message.reasoning)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about vLLM."}
    ],
    temperature=0.2,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print("Content:", resp2.choices[0].message.content)

Reasoning on
Reasoning: Here's a thinking process:

1.  **Analyze the Request:**
   - User wants a haiku about GPUs.
   - A haiku has specific structure: 3 lines, 5 syllables, 7 syllables, 5 syllables (total 17 syllables).
   - Topic: GPUs (Graphics Processing Units).

2.  **Identify Key Concepts about GPUs:**
   - Parallel processing
   - Graphics/gaming
   - Deep learning/AI
   - Chips/boards
   - Fast computation
   - Many cores
   - Rendering pixels

3.  **Brainstorm Syllable Patterns (5-7-5):**
   I need to come up with three lines that follow 5-7-5 and relate to GPUs.

   Let's try to draft:

   Line 1 (5 syllables): 
   - "Silicon cores" (3) - no
   - "Graphic chips" (2) - no
   - "Chip that glows" (3) - no
   - "Parallel" (3) - no
   - "Compute fast" (2) - no
   - "Processing" (3) - no
   Let's count carefully:
   - "Silicon valleys" (4) - no
   - "Graphics card" (3) - no
   - "Chip within" (2) - no
   Let's try to build around 5 syllables:
   - "Parallel cores spin" -> P a r a

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`.

In [9]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=False
)

choice = completion.choices[0]
print("Reasoning:", choice.message.reasoning)
print("Tool calls:", choice.message.tool_calls)

Reasoning: Here's a thinking process:

1.  **Analyze User Input:**
   - Bill total: $50
   - Tip percentage: 15%
   - Question: What will be the amount for 15% tip?

2.  **Identify Required Tool:**
   - The `calculate_tip` function takes `bill_total` and `tip_percentage` as integers.
   - I need to call this function with the given values.

3.  **Check Parameters:**
   - `bill_total`: 50 (integer)
   - `tip_percentage`: 15 (integer)
   - Both are required and match the input.

4.  **Call the Function:**
   - Use the exact format specified.
   - `calculate_tip(bill_total=50, tip_percentage=15)`

5.  **Execute/Output:**
   - I'll generate the function call.✅
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-a48259a7c25ed5dc', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]


### Controlling Reasoning Budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [11]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [13]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"  # use actual HF model ID for tokenizer
)

In [14]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    max_tokens=1024,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: Here, the user wants a haiku about GPUs. I need to craft a haiku that follows the 5-7-5 syllable structure, focusing on GPUs (Graphics Processing Units). A haiku typically focuses on nature, but can be about any subject. I should capture the essence of GPUs: parallel processing, graphics, gaming, computation, cores, etc.

Let me brainstorm some lines:

Line 1 (5 syllables): Maybe "Silicon minds" (2 syllables? "Sil-i-con minds" -> 4? Actually "Silicon" is 3, "minds" is 1,.
Content: Silicon minds
Parallel dreams on screen
Graphics unfold


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the Docker container in the terminal where it was started (`Ctrl+C`). The `--rm` flag automatically removes the container on exit.
2. Restart the kernel if needed to ensure a clean state.